In [9]:
# -*- coding: utf-8 -*-
"""
Generador de mapa KMZ desde Excel para Google My Maps
Funciona en Google Colab - sin instalar nada localmente
"""

# Instalar solo la librería que no viene por defecto en Colab
!pip install simplekml -q

import pandas as pd
import simplekml
import requests
import time
import os
from google.colab import files
from IPython.display import display, HTML

# ========== CONFIGURACIÓN ==========
# Coloca aquí tu API Key de Google (si vas a usar geocodificación)
API_KEY = ""   # Déjalo vacío si tus coordenadas ya están en el Excel

# ========== FUNCIONES ==========
def geocodificar(direccion, api_key):
    if not api_key or not direccion:
        return None, None
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": direccion, "key": api_key}
    try:
        resp = requests.get(url, params=params).json()
        if resp["status"] == "OK":
            loc = resp["results"][0]["geometry"]["location"]
            return loc["lat"], loc["lng"]
    except:
        pass
    return None, None

def crear_html_popup(fila, nombre_col):
    html = f"<div style='font-family:Arial'><b>{fila[nombre_col]}</b><br/><hr/>"
    for col, val in fila.items():
        if col != nombre_col and pd.notna(val):
            html += f"<b>{col}:</b> {val}<br/>"
    html += "</div>"
    return html

# ========== SUBIR ARCHIVO EXCEL ==========
print("📂 Sube tu archivo Excel (debe tener columnas 'Nombre Comercial' y opcionalmente 'Latitud'/'Longitud' o 'Dirección')")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)
print(f"✅ Archivo cargado: {filename}")
print(f"📊 Columnas encontradas: {list(df.columns)}")

# ========== PROCESAR ==========
kml = simplekml.Kml(name="Mi Mapa")
kml.document.name = "Generado desde Excel"

marcadores_ok = 0
errores = 0

for idx, row in df.iterrows():
    nombre = row.get("Nombre Comercial", f"Punto {idx+1}")
    print(f"Procesando {idx+1}: {nombre}")

    # Buscar coordenadas
    lat, lon = None, None
    if "Latitud" in df.columns and "Longitud" in df.columns:
        try:
            lat = float(row["Latitud"])
            lon = float(row["Longitud"])
        except:
            pass

    if (lat is None or lon is None) and "Dirección" in df.columns and API_KEY:
        direc = row["Dirección"]
        if pd.notna(direc):
            lat, lon = geocodificar(direc, API_KEY)
            time.sleep(0.2)

    if lat is None or lon is None:
        print(f"  ⚠️ No se pudo ubicar, se omite")
        errores += 1
        continue

    # Crear marcador
    p = kml.newpoint(name=str(nombre)[:200], coords=[(lon, lat)])
    p.description = crear_html_popup(row, "Nombre Comercial")
    p.style.iconstyle.color = simplekml.Color.green
    p.style.iconstyle.scale = 1.2
    marcadores_ok += 1

# ========== GUARDAR Y DESCARGAR ==========
nombre_kmz = "mi_mapa.kmz"
kml.savekmz(nombre_kmz)
print(f"\n🎉 Mapa generado con {marcadores_ok} marcadores. {errores} omitidos.")
files.download(nombre_kmz)
print("✅ Descarga iniciada. Guarda el archivo .kmz en tu computadora.")

📂 Sube tu archivo Excel (debe tener columnas 'Nombre Comercial' y opcionalmente 'Latitud'/'Longitud' o 'Dirección')


Saving Irapuato.xlsx to Irapuato.xlsx
✅ Archivo cargado: Irapuato.xlsx
📊 Columnas encontradas: ['No', 'Nombre Comercial', 'Ciudad', 'Estado', 'Latitud', 'Longitud', 'Visita']
Procesando 1: ELSA RODRIGUEZ ZARATE
Procesando 2: FERRETOMELOPEZ
Procesando 3: ROBERTO ACOSTA RODRIGUEZ
Procesando 4: STEPHANI AMALIA BARRIO NEGRETE
Procesando 5: KARLA ANDREA PATIÑO CERVANTES
Procesando 6: MATERIALES FERRETANA
Procesando 7: KARINA GABRIELA SANCHEZ
Procesando 8: JUAN CARLOS RIVERA
Procesando 9: ERIK EDUARDO FUENTES ALONSO
Procesando 10: MARIA DE JESUS ALONSO CISNEROS
Procesando 11: EMILIO ARENAS SANTOS
Procesando 12: CARLOS EDUARDO MARTINEZ OCTAVIO
Procesando 13: JOSE MARTIN DELGADO PEREZ
Procesando 14: JOAN CRISTIAN CORDOBA ARTEAGA
Procesando 15: ADRIANA HERNANDEZ ALVARES
Procesando 16: JUAN CARLOS GARCIA RIVERA
Procesando 17: MIGUEL ALONSO CISNEROS VPG
Procesando 18: ALBERTO SANCHEZ SOLORZANO
Procesando 19: HUGO REA VAZQUEZ
Procesando 20: ANA BERTHA VILLAGOMEZ CALIXTO
Procesando 21: EDUARDO CERV

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Descarga iniciada. Guarda el archivo .kmz en tu computadora.
